# 🐍 Python Tutorial med fokus på AI-103
### Azure AI App & Agent Developer Associate

---

Den här notebooken täcker:
- Grundläggande Python-koncept som du behöver för att klara **AI-103**
- Praktiska kodexempel för varje **Azure AI-tjänst** som ingår i examen
- Kommentarer på svenska med kodexempel på engelska (branschstandard)

**Kapitel:**
1. 🔁 Loops
2. 🔀 Conditionals
3. 📦 Variables & Types
4. 🧩 Functions
5. 🏗️ OOP
6. 🚨 Error Handling
7. ⚡ Async & APIs
8. 🌐 Azure AI Foundry
9. 🤖 Generative AI & Agents
10. 👁️ Computer Vision
11. 📝 Text & Language
12. 📄 Search & Extraction
13. 🔒 Security & MLOps

---
> 📝 **Tips:** Kör varje cell med `Shift + Enter`. Börja uppifrån och jobba dig nedåt.

©️ Stefan Blecko 2026

---
## 🔁 1. Loops
Loopar används för att **upprepa kod** ett visst antal gånger eller tills ett villkor är uppfyllt.  
I AI-103-sammanhang används loopar ofta för att iterera över sökresultat, dokumentfält eller API-svar.

### `for`-loop
Itererar över ett intervall eller en lista.

In [ ]:
# Grundläggande for-loop med range()
for i in range(5):
    print(f"Steg {i}")

print("---")

# Iterera över en lista (vanligt vid AI-svar)
azure_services = ["OpenAI", "Vision", "Language", "Search", "Document Intelligence"]
for service in azure_services:
    print(f"Azure AI-tjänst: {service}")

### `while`-loop
Upprepar så länge ett villkor är sant — användbart vid polling av API-status.

In [ ]:
# Simulerad polling: väntar tills en operation är klar
import time

status = "running"
attempts = 0

while status != "succeeded":
    attempts += 1
    print(f"Försök {attempts}: status = {status}")
    if attempts >= 3:
        status = "succeeded"  # simulerar att jobbet är klart

print("✅ Operation klar!")

### `break` och `continue`
- `break` avslutar loopen direkt
- `continue` hoppar till nästa iteration

In [ ]:
results = ["doc1", "doc2", None, "doc4", "doc5"]

for doc in results:
    if doc is None:
        print("Tomt resultat – hoppar över")
        continue  # hoppa över None
    if doc == "doc4":
        print("Hittade doc4 – avslutar sökning")
        break     # stoppa vid doc4
    print(f"Bearbetar: {doc}")

### Nästlade loopar
Vanliga vid bearbetning av sidor och rader i dokument (t.ex. Document Intelligence).

In [ ]:
# Simulerat OCR-resultat: sidor med rader
pages = [
    ["Faktura nr: 1001", "Datum: 2026-01-15"],
    ["Summa: 4 500 kr",  "Moms: 900 kr"]
]

for page_num, page in enumerate(pages, start=1):
    print(f"\n📄 Sida {page_num}:")
    for line in page:
        print(f"   {line}")

---
## 🔀 2. Conditionals
Villkorssatser styr **flödet** i koden baserat på värden.  
Kritiskt för att hantera API-svar, felkoder och konfidenspoäng från AI-tjänster.

### `if` / `elif` / `else`

In [ ]:
# Klassificera konfidenspoäng från en AI-modell
confidence = 0.82

if confidence >= 0.90:
    label = "Hög konfidans ✅"
elif confidence >= 0.70:
    label = "Medel konfidans ⚠️"
else:
    label = "Låg konfidans ❌"

print(f"Konfidans: {confidence:.0%} → {label}")

### Ternary-operator (inline villkor)

In [ ]:
sentiment_score = 0.75
result = "positiv" if sentiment_score > 0.5 else "negativ"
print(f"Sentiment: {result}")

### `match` / `case` (Python 3.10+)
Ersätter långa if-elif-kedjor — t.ex. för att hantera HTTP-statuskoder från Azure REST API.

In [ ]:
def handle_status(code: int) -> str:
    match code:
        case 200: return "OK – lyckades"
        case 201: return "Created – resurs skapad"
        case 400: return "Bad Request – kontrollera parametrar"
        case 401: return "Unauthorized – kontrollera API-nyckel"
        case 429: return "Too Many Requests – rate limit"
        case 500: return "Internal Server Error – försök igen"
        case _:   return f"Okänd statuskod: {code}"

for code in [200, 401, 429, 503]:
    print(f"{code}: {handle_status(code)}")

### Boolean-logik och short-circuit

In [ ]:
has_key    = True
has_endpoint = True
is_region_supported = False

# and kräver att ALLA är sanna
if has_key and has_endpoint and is_region_supported:
    print("Redo att ansluta")
else:
    print("Konfiguration saknas – kontrollera region och autentisering")

# or – minst ett villkor
use_managed_identity = False
use_api_key = True
can_auth = use_managed_identity or use_api_key
print(f"Kan autentisera: {can_auth}")

---
## 📦 3. Variables & Types
Python är dynamiskt typat men AI-103 SDK:er använder ofta **typannoteringar** för tydlighet.

In [ ]:
# Primitiva typer
endpoint: str   = "https://my-ai.cognitiveservices.azure.com/"
max_tokens: int = 1024
temperature: float = 0.7
streaming: bool = True
no_value = None  # saknar värde

print(type(endpoint), type(max_tokens), type(temperature))

In [ ]:
# Lista – ordnad, muterbar
messages = [
    {"role": "system",  "content": "Du är en hjälpsam assistent."},
    {"role": "user",    "content": "Vad är Azure AI Foundry?"}
]

# Dictionary – nyckel-värde-par
config = {
    "model": "gpt-4o",
    "temperature": 0.7,
    "max_tokens": 512
}

# Tupel – ordnad, omuterbar
supported_regions = ("eastus", "westeurope", "swedencentral")

# Set – unika värden
features = {"ocr", "caption", "objects", "ocr"}  # duplicat tas bort

print(f"Meddelanden: {len(messages)} st")
print(f"Modell: {config['model']}")
print(f"Regioner: {supported_regions}")
print(f"Features (unika): {features}")

In [ ]:
# Typkonvertering – vanligt vid JSON-svar från API
raw_score = "0.93"         # str från JSON
score = float(raw_score)   # konvertera till float
pct   = int(score * 100)   # till int

print(f"Rå: '{raw_score}' ({type(raw_score).__name__})")
print(f"Float: {score} | Int: {pct}%")

---
## 🧩 4. Functions
Funktioner är grunden för **återanvändbar kod**.  
I AI-103 skriver du hjälpfunktioner för autentisering, API-anrop och resultathantering.

In [ ]:
# Grundläggande funktion med typannoteringar
def format_endpoint(base_url: str, path: str) -> str:
    """Kombinerar bas-URL och sökväg till en komplett endpoint."""
    return f"{base_url.rstrip('/')}/{path.lstrip('/')}"

url = format_endpoint("https://my-ai.cognitiveservices.azure.com", "/openai/deployments")
print(url)

In [ ]:
# Standardargument och keyword arguments
def build_message(content: str, role: str = "user") -> dict:
    return {"role": role, "content": content}

msg1 = build_message("Hej!")                        # role = "user" (standard)
msg2 = build_message("Var hjälpsam.", role="system") # override
print(msg1)
print(msg2)

In [ ]:
# Lambda – kompakt anonym funktion
# Vanlig vid sortering av resultat
results = [
    {"doc": "A", "score": 0.72},
    {"doc": "B", "score": 0.91},
    {"doc": "C", "score": 0.65},
]

sorted_results = sorted(results, key=lambda r: r["score"], reverse=True)
for r in sorted_results:
    print(f"{r['doc']}: {r['score']:.0%}")

In [ ]:
# Rekursion – beräkna token-kostnad exponentiellt (pedagogiskt exempel)
def token_cost(tokens: int, price_per_1k: float = 0.002, depth: int = 0) -> float:
    """Beräknar kostnad rekursivt (illustrativt exempel)."""
    if tokens <= 0:
        return 0.0
    chunk = min(tokens, 1000)
    return (chunk / 1000 * price_per_1k) + token_cost(tokens - chunk, price_per_1k, depth + 1)

total = token_cost(3500)
print(f"Kostnad för 3 500 tokens: ${total:.4f}")

---
## 🏗️ 5. OOP – Objektorienterad programmering
Azure AI SDK:er är uppbyggda kring klasser. Du instansierar klienter, anropar metoder och hanterar objekt.

In [ ]:
# Grundläggande klass – modellerar en Azure AI-klient
class AIClient:
    """Bas-klass för Azure AI-klienter."""

    def __init__(self, endpoint: str, api_key: str):
        self.endpoint = endpoint
        self.api_key  = api_key
        self._connected = False

    def connect(self) -> None:
        """Simulerar anslutning."""
        self._connected = True
        print(f"✅ Ansluten till {self.endpoint}")

    def status(self) -> str:
        return "ansluten" if self._connected else "frånkopplad"


client = AIClient("https://my-ai.cognitiveservices.azure.com", "abc123")
print(f"Status: {client.status()}")
client.connect()
print(f"Status: {client.status()}")

In [ ]:
# Arv – specialiserade klienter ärver från AIClient
class VisionClient(AIClient):
    """Klient specifik för Azure AI Vision."""

    def analyze(self, image_url: str) -> dict:
        if not self._connected:
            raise RuntimeError("Anslut först med connect()")
        # Simulerar API-anrop
        return {"caption": "En laptop på ett skrivbord", "confidence": 0.95}


class LanguageClient(AIClient):
    """Klient specifik för Azure AI Language."""

    def sentiment(self, text: str) -> str:
        # Förenklad sentiment-analys
        positive_words = {"bra", "fantastisk", "utmärkt", "great", "excellent"}
        words = set(text.lower().split())
        return "positiv" if words & positive_words else "neutral/negativ"


vision   = VisionClient("https://vision.azure.com", "key-v")
language = LanguageClient("https://lang.azure.com",   "key-l")

vision.connect()
result = vision.analyze("https://example.com/photo.jpg")
print(f"Bildanalys: {result['caption']} ({result['confidence']:.0%})")

language.connect()
print(f"Sentiment: {language.sentiment('Det här är en fantastisk tjänst!')}")

In [ ]:
# Dataclass – kompakt sätt att definiera datamodeller (vanligt i AI-pipelines)
from dataclasses import dataclass, field
from typing import List

@dataclass
class ChatMessage:
    role: str
    content: str

@dataclass
class Conversation:
    messages: List[ChatMessage] = field(default_factory=list)

    def add(self, role: str, content: str) -> None:
        self.messages.append(ChatMessage(role, content))

    def to_api_format(self) -> list:
        return [{"role": m.role, "content": m.content} for m in self.messages]


conv = Conversation()
conv.add("system", "Du är en hjälpsam Azure-expert.")
conv.add("user",   "Vad är RAG?")

for msg in conv.to_api_format():
    print(f"[{msg['role']}] {msg['content']}")

---
## 🚨 6. Error Handling
Robust felhantering är **kritisk** i AI-applikationer. API-anrop kan misslyckas pga. nätverksfel, rate limits eller ogiltiga parametrar.

In [ ]:
import json

# try / except / else / finally
def parse_api_response(raw: str) -> dict:
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"❌ Ogiltigt JSON: {e}")
        return {}
    except Exception as e:
        print(f"❌ Oväntat fel: {e}")
        return {}
    else:
        print("✅ JSON parsad utan fel")
        return data
    finally:
        print("   (parse_api_response avslutad)")

good = parse_api_response('{"sentiment": "positive", "score": 0.91}')
print(good)

bad  = parse_api_response('det här är inte JSON')
print(bad)

In [ ]:
# Egna undantagsklasser – professionell praxis
class AzureAIError(Exception):
    """Bas-undantag för Azure AI-fel."""
    pass

class RateLimitError(AzureAIError):
    """Kastas vid HTTP 429."""
    def __init__(self, retry_after: int = 60):
        self.retry_after = retry_after
        super().__init__(f"Rate limit. Försök igen om {retry_after} sekunder.")

class AuthError(AzureAIError):
    """Kastas vid HTTP 401."""
    pass


def simulate_api_call(status_code: int):
    if status_code == 429:
        raise RateLimitError(retry_after=30)
    if status_code == 401:
        raise AuthError("Ogiltig API-nyckel – kontrollera konfigurationen.")
    return {"status": "ok"}


for code in [200, 401, 429]:
    try:
        res = simulate_api_call(code)
        print(f"HTTP {code}: {res}")
    except RateLimitError as e:
        print(f"⏳ HTTP {code}: {e} (retry_after={e.retry_after}s)")
    except AuthError as e:
        print(f"🔑 HTTP {code}: {e}")

In [ ]:
# Retry-logik med exponentiell backoff (mönster för produktionskod)
import time, random

def with_retry(fn, max_attempts: int = 3):
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except RateLimitError as e:
            wait = 2 ** attempt + random.uniform(0, 1)
            print(f"⏳ Försök {attempt}/{max_attempts} misslyckades. Väntar {wait:.1f}s…")
            if attempt == max_attempts:
                raise
            time.sleep(0.1)  # kortad ned för demo

call_count = {"n": 0}
def flaky_call():
    call_count["n"] += 1
    if call_count["n"] < 3:
        raise RateLimitError()
    return "✅ Lyckades!"

result = with_retry(flaky_call)
print(result)

---
## ⚡ 7. Async & APIs
Azure AI SDK:er stöder **asynkrona anrop** för bättre prestanda i produktionskod.  
Asynkron programmering låter dig hantera flera anrop parallellt utan att blockera.

In [ ]:
import asyncio

# Grundläggande async/await
async def analyze_document(doc_id: str) -> dict:
    """Simulerar ett asynkront API-anrop."""
    await asyncio.sleep(0.1)  # I verkligheten: nätverksanrop
    return {"doc_id": doc_id, "status": "analyserad", "pages": 3}

async def main():
    result = await analyze_document("DOC-001")
    print(result)

await main()  # I JupyterLab fungerar await direkt i cellen

In [ ]:
# Parallella anrop med asyncio.gather – analysera flera dokument samtidigt
async def process_batch(doc_ids: list) -> list:
    tasks = [analyze_document(doc_id) for doc_id in doc_ids]
    results = await asyncio.gather(*tasks)
    return list(results)

docs = ["DOC-001", "DOC-002", "DOC-003", "DOC-004"]
results = await process_batch(docs)

for r in results:
    print(f"📄 {r['doc_id']}: {r['status']} ({r['pages']} sidor)")

In [ ]:
# REST API-anrop med requests (synkront – enklare för scripting)
# OBS: Kräver pip install requests
# pip install requests

import json

def call_azure_api(endpoint: str, api_key: str, payload: dict) -> dict:
    """
    Mönster för REST-anrop mot Azure AI.
    I verkligheten: ersätt med requests.post()
    """
    headers = {
        "Content-Type":            "application/json",
        "Ocp-Apim-Subscription-Key": api_key,
    }
    # Simulerat svar
    print(f"POST {endpoint}")
    print(f"Headers: {list(headers.keys())}")
    print(f"Payload: {json.dumps(payload, ensure_ascii=False)}")
    return {"status": "ok", "result": "simulerat svar"}

response = call_azure_api(
    endpoint="https://my-lang.cognitiveservices.azure.com/text/analytics/v3.1/sentiment",
    api_key="<DIN-NYCKEL>",
    payload={"documents": [{"id": "1", "text": "Azure AI är fantastiskt!", "language": "sv"}]}
)
print(f"\nSvar: {response}")

---
# 🌐 Azure AI Domains
## 🌐 8. Azure AI Foundry
**Azure AI Foundry** (tidigare Azure AI Studio) är den centrala plattformen för att bygga, driftsätta och hantera AI-applikationer på Azure.

### Viktigt för AI-103:
| Koncept | Beskrivning |
|---|---|
| **Hub & Project** | Hierarki för att organisera resurser |
| **Model catalog** | Välj mellan OpenAI, Meta, Mistral m.fl. |
| **Prompt flow** | Visuellt verktyg för LLM-pipelines |
| **Evaluation** | Mät kvalitet: relevance, groundedness, fluency |
| **Content Safety** | Filtrera skadligt innehåll |
| **Responsible AI** | Fairness, transparency, accountability |

In [ ]:
# ─── Installation (kör en gång) ───────────────────────────────────────────────
# !pip install azure-ai-projects azure-identity openai

# Konfiguration – ersätt med dina värden
SUBSCRIPTION_ID   = "<DIN-SUBSCRIPTION-ID>"
RESOURCE_GROUP    = "<DIN-RESOURCE-GROUP>"
PROJECT_NAME      = "<DITT-PROJEKT"
AZURE_ENDPOINT    = "https://<DITT-NAMN>.openai.azure.com/"
AZURE_API_KEY     = "<DIN-API-NYCKEL>"
API_VERSION       = "2024-08-01-preview"

print("✅ Konfigurationsvariablerna är satta (fyll i dina egna värden)")

In [ ]:
# Content Safety – filtrera skadligt innehåll
# !pip install azure-ai-contentsafety

# from azure.ai.contentsafety import ContentSafetyClient
# from azure.ai.contentsafety.models import AnalyzeTextOptions
# from azure.core.credentials import AzureKeyCredential

# Simulerat exempel (kommentera bort ovan och fyll i riktiga värden i produktion)
def simulate_content_safety(text: str) -> dict:
    """Simulerar Content Safety API-svar."""
    # Skadekategorier: Hate, SelfHarm, Sexual, Violence
    # Allvarlighetsnivåer: 0 (ingen), 2 (låg), 4 (medel), 6 (hög)
    return {
        "hate":      {"severity": 0},
        "self_harm": {"severity": 0},
        "sexual":    {"severity": 0},
        "violence":  {"severity": 0},
    }

text_to_check = "Azure AI är ett fantastiskt verktyg för företag."
safety_result = simulate_content_safety(text_to_check)

is_safe = all(cat["severity"] == 0 for cat in safety_result.values())
print(f"Text: '{text_to_check}'")
print(f"Säker: {'✅ Ja' if is_safe else '❌ Nej'}")
for category, result in safety_result.items():
    print(f"  {category}: allvarlighet = {result['severity']}")

---
## 🤖 9. Generative AI & Agents
Kärnan i AI-103. Du ska kunna bygga **chattapplikationer**, **RAG-pipelines** och **agenter** med Azure OpenAI.

### Viktiga koncept:
- `chat.completions.create()` – grundläggande LLM-anrop
- **Prompt engineering** – system, user, assistant-roller
- **RAG** – Retrieval-Augmented Generation
- **Function calling / Tool use** – låt LLM anropa funktioner
- **Streaming** – visa svar token för token

In [ ]:
# Grundläggande Azure OpenAI-anrop
# !pip install openai

# from openai import AzureOpenAI

# client = AzureOpenAI(
#     azure_endpoint = AZURE_ENDPOINT,
#     api_key        = AZURE_API_KEY,
#     api_version    = API_VERSION
# )

# Simulerat anrop (ta bort kommentarerna ovan för riktiga anrop)
def simulate_chat_completion(messages: list, model: str = "gpt-4o", **kwargs) -> str:
    """Simulerar ett Azure OpenAI chat.completions.create() svar."""
    user_msg = next((m["content"] for m in messages if m["role"] == "user"), "")
    return f"[Simulerat svar på: '{user_msg}']"

messages = [
    {"role": "system", "content": "Du är en expert på Azure AI-tjänster."},
    {"role": "user",   "content": "Förklara vad RAG är med ett enkelt exempel."}
]

response = simulate_chat_completion(messages, model="gpt-4o", temperature=0.7, max_tokens=512)
print(f"Svar: {response}")

In [ ]:
# RAG-pipeline – Retrieval-Augmented Generation
# Steg 1: Embed frågan → Steg 2: Sök index → Steg 3: Skicka kontext till LLM

def embed_text(text: str) -> list:
    """Simulerar text-embedding (i verkligheten: Azure OpenAI Embeddings)."""
    return [hash(word) % 1000 / 1000 for word in text.split()[:5]]  # falsk vektor

def vector_search(query_vector: list, top_k: int = 3) -> list:
    """Simulerar sökning i Azure AI Search."""
    return [
        {"id": "doc1", "content": "RAG kombinerar sökning med generering för bättre svar.", "score": 0.95},
        {"id": "doc2", "content": "Azure AI Search stöder vektorsökning och semantisk rankning.", "score": 0.88},
        {"id": "doc3", "content": "Embeddings omvandlar text till numeriska vektorer.", "score": 0.81},
    ][:top_k]

def rag_query(user_question: str) -> str:
    # 1. Embed frågan
    query_vector = embed_text(user_question)

    # 2. Hämta relevanta dokument
    docs = vector_search(query_vector, top_k=3)
    context = "\n".join(f"- {d['content']}" for d in docs)

    # 3. Bygg prompt med kontext
    messages = [
        {"role": "system", "content": f"Svara ENBART baserat på följande kontext:\n{context}"},
        {"role": "user",   "content": user_question}
    ]

    # 4. Anropa LLM
    return simulate_chat_completion(messages)

print("RAG Pipeline Demo")
print("=" * 40)
question = "Vad är RAG och hur fungerar det?"
answer   = rag_query(question)
print(f"Fråga: {question}")
print(f"Svar:  {answer}")

In [ ]:
# Function Calling / Tool Use
import json

# Definiera verktyg som LLM:en kan anropa
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_azure_service_info",
            "description": "Hämtar information om en Azure AI-tjänst",
            "parameters": {
                "type": "object",
                "properties": {
                    "service_name": {
                        "type": "string",
                        "description": "Namnet på Azure AI-tjänsten, t.ex. 'Azure OpenAI'"
                    }
                },
                "required": ["service_name"]
            }
        }
    }
]

# Implementera funktionen
def get_azure_service_info(service_name: str) -> dict:
    services = {
        "Azure OpenAI": {"type": "Generative AI", "models": ["gpt-4o", "text-embedding-ada-002"]},
        "Azure AI Vision": {"type": "Computer Vision", "features": ["OCR", "Caption", "Objects"]},
    }
    return services.get(service_name, {"error": "Tjänst ej hittad"})

# Simulera tool_call-flöde
tool_call = {"name": "get_azure_service_info", "arguments": '{"service_name": "Azure OpenAI"}'}
args   = json.loads(tool_call["arguments"])
result = get_azure_service_info(**args)

print(f"Tool: {tool_call['name']}")
print(f"Args: {args}")
print(f"Resultat: {json.dumps(result, indent=2, ensure_ascii=False)}")

---
## 👁️ 10. Computer Vision
**Azure AI Vision** erbjuder bildanalys, OCR och objektidentifiering.  
I AI-103 testas du på hur du integrerar dessa tjänster via SDK.

In [ ]:
# Azure AI Vision – Bildanalys
# !pip install azure-ai-vision-imageanalysis

# Produktion:
# from azure.ai.vision.imageanalysis import ImageAnalysisClient
# from azure.ai.vision.imageanalysis.models import VisualFeatures
# from azure.core.credentials import AzureKeyCredential

# Simulerat svar
def simulate_image_analysis(image_url: str) -> dict:
    return {
        "caption": {
            "text":       "En person som arbetar vid en laptop i ett ljust kontor",
            "confidence": 0.94
        },
        "objects": [
            {"name": "laptop",  "confidence": 0.98, "bounding_box": {"x": 100, "y": 200, "w": 300, "h": 200}},
            {"name": "person",  "confidence": 0.96, "bounding_box": {"x":  10, "y":  50, "w": 150, "h": 400}},
            {"name": "monitor", "confidence": 0.87, "bounding_box": {"x": 400, "y": 100, "w": 200, "h": 150}},
        ],
        "tags": ["teknik", "arbete", "inomhus", "dator"],
        "read": {"lines": [{"text": "Azure AI Vision"}]}
    }

image_url = "https://example.com/office.jpg"
result    = simulate_image_analysis(image_url)

print(f"📷 Bildanalys: {image_url}")
print(f"\n📝 Bildtext: {result['caption']['text']}")
print(f"   Konfidans: {result['caption']['confidence']:.0%}")

print("\n🔍 Identifierade objekt:")
for obj in result["objects"]:
    print(f"   - {obj['name']} ({obj['confidence']:.0%})")

print(f"\n🏷️ Taggar: {', '.join(result['tags'])}")
print(f"\n📄 OCR-text: {result['read']['lines'][0]['text']}")

In [ ]:
# Produktion – fullständigt SDK-mönster (kommentera in när du har riktiga credentials)
VISION_ENDPOINT = "https://<DITT-NAMN>.cognitiveservices.azure.com/"
VISION_KEY      = "<DIN-VISION-NYCKEL>"

sdk_code = '''
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential

client = ImageAnalysisClient(
    endpoint   = VISION_ENDPOINT,
    credential = AzureKeyCredential(VISION_KEY)
)

result = client.analyze_from_url(
    image_url       = "https://example.com/photo.jpg",
    visual_features = [
        VisualFeatures.CAPTION,
        VisualFeatures.OBJECTS,
        VisualFeatures.READ,
        VisualFeatures.TAGS
    ]
)

print(result.caption.text)
for line in result.read.blocks[0].lines:
    print(line.text)
'''

print("Produktion SDK-kod:")
print(sdk_code)

---
## 📝 11. Text & Language
**Azure AI Language** erbjuder NLP-funktioner: sentiment, entiteter, nyckelfraser, sammanfattning och mer.

**Azure AI Speech** erbjuder tal-till-text och text-till-tal.

In [ ]:
# Claude Sonnet 5 Extra test 2026-07-10
import os                                                         # för att läsa miljövariabler (API-nycklar, endpoint)
from azure.ai.inference import ChatCompletionsClient              # klienten som pratar med Microsoft Foundry
from azure.ai.inference.models import SystemMessage, UserMessage  # meddelandetyper till chat-modellen
from azure.core.credentials import AzureKeyCredential             # hanterar autentisering med API-nyckel


def flom_load_ai(endpoint=None, model="gpt-4o-mini"):
    """Genererar en rolig, politiskt klingande floskelmening via Microsoft Foundry."""
    # Läs endpoint från argument, annars från miljövariabeln FOUNDRY_ENDPOINT
    endpoint = endpoint or os.environ["FOUNDRY_ENDPOINT"]
    # Läs API-nyckeln från miljövariabeln FOUNDRY_API_KEY (aldrig hårdkodad i koden)
    api_key = os.environ["FOUNDRY_API_KEY"]

    # Skapa en klient som kan skicka förfrågningar till Foundry-endpointen
    client = ChatCompletionsClient(
        endpoint=endpoint,                         # URL till Foundry-deploymenten
        credential=AzureKeyCredential(api_key)     # autentisering via nyckel
    )

    # Skicka en chat-förfrågan till modellen och vänta på svar
    response = client.complete(
        model=model,                               # namnet på modell-deploymenten i Foundry
        messages=[
            SystemMessage(content=(                # systeminstruktion: styr TON och FORMAT på svaret
                "Du genererar korta, satiriska svenska politiska floskler "
                "i formen 'Vi <parti> <predikat> <objekt>.' "
                "Svara endast med meningen, inget annat."
            )),
            UserMessage(content="Generera en ny floskelmening.")  # själva "beställningen" till modellen
        ],
        temperature=1.0,                           # högt värde = mer slumpmässiga/kreativa svar
        max_tokens=40,                              # begränsar svarets längd (kort mening räcker)
    )

    # Plocka ut textsvaret ur API-svaret och trimma bort onödiga mellanslag/radbrytningar
    sentence = response.choices[0].message.content.strip()
    return sentence                                 # skicka tillbaka den färdiga meningen


# --- Inställningar (redigera direkt här istället för kommandoradsargument) ---
ANTAL_FLOSKLER = 5              # hur många floskler som ska genereras i den här cellen
ENDPOINT = None                 # ange din Foundry-endpoint här, eller låt vara None för att läsa FOUNDRY_ENDPOINT
MODEL = "gpt-4o-mini"           # vilken modell-deployment i Foundry som ska användas

# Generera och skriv ut floskler direkt i cellens output
for _ in range(ANTAL_FLOSKLER):
    print(flom_load_ai(endpoint=ENDPOINT, model=MODEL))

ModuleNotFoundError: No module named 'azure'

In [ ]:
# Entitetsigenkänning – Named Entity Recognition (NER)
def simulate_ner(text: str) -> list:
    """Simulerar Azure AI Language NER-svar."""
    return [
        {"text": "Microsoft",    "category": "Organization", "confidence": 0.99},
        {"text": "Azure",        "category": "Product",      "confidence": 0.97},
        {"text": "Stockholm",    "category": "Location",     "confidence": 0.95},
        {"text": "2024",         "category": "DateTime",     "confidence": 0.98},
    ]

text = "Microsoft lanserade Azure AI Foundry i Stockholm under 2024."
entities = simulate_ner(text)

print(f"Text: '{text}'")
print("\nIdentifierade entiteter:")
for e in entities:
    print(f"  [{e['category']:12s}] '{e['text']}' ({e['confidence']:.0%})")

In [ ]:
# Azure AI Translator – Mönster
TRANSLATOR_KEY      = "<DIN-TRANSLATOR-NYCKEL>"
TRANSLATOR_ENDPOINT = "https://api.cognitive.microsofttranslator.com"
TRANSLATOR_REGION   = "swedencentral"

translation_code = '''
import requests, uuid

path    = "/translate"
params  = {"api-version": "3.0", "from": "sv", "to": ["en", "de"]}
headers = {
    "Ocp-Apim-Subscription-Key":    TRANSLATOR_KEY,
    "Ocp-Apim-Subscription-Region": TRANSLATOR_REGION,
    "Content-Type":                 "application/json",
    "X-ClientTraceId":              str(uuid.uuid4())
}
body = [{"text": "Hej världen! Azure AI är fantastiskt."}]

response = requests.post(
    TRANSLATOR_ENDPOINT + path,
    params=params, headers=headers, json=body
)
translations = response.json()[0]["translations"]
for t in translations:
    print(f"{t[\"to\"]}: {t[\"text\"]}")  
'''

print("Azure AI Translator – SDK-mönster:")
print(translation_code)

# Simulerat svar
print("Simulerat svar:")
print("en: Hello world! Azure AI is fantastic.")
print("de: Hallo Welt! Azure AI ist fantastisch.")

---
## 📄 12. Search & Extraction
Två nyckelkomponenter i AI-103:
- **Azure AI Search** – Fulltext-, vektor- och hybridsökning
- **Azure AI Document Intelligence** – Extrahera strukturerad data från dokument

In [ ]:
# Azure AI Search – Sökmönster
# !pip install azure-search-documents

# Simulerat svar
def simulate_search(query: str, top: int = 3) -> list:
    """Simulerar Azure AI Search-svar."""
    mock_docs = [
        {"id": "1", "title": "Introduktion till RAG",        "content": "RAG kombinerar retrieval och generation...", "score": 0.95},
        {"id": "2", "title": "Azure AI Search och vektorer",  "content": "Vektorsökning möjliggör semantisk sökning...", "score": 0.89},
        {"id": "3", "title": "Prompt engineering för AI-103", "content": "Effektiva prompts förbättrar AI-svar...", "score": 0.82},
        {"id": "4", "title": "Azure OpenAI Service guide",    "content": "Driftsätt GPT-modeller på Azure...", "score": 0.75},
    ]
    return sorted(mock_docs, key=lambda x: x["score"], reverse=True)[:top]

results = simulate_search("vektorsökning RAG", top=3)

print(f"🔍 Sökresultat:")
print("-" * 55)
for i, doc in enumerate(results, 1):
    print(f"{i}. [{doc['score']:.2f}] {doc['title']}")
    print(f"   {doc['content'][:55]}…")

In [ ]:
# Skapa ett sökindex med vektorfält – SDK-mönster
index_schema = '''
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField,
    SearchField, SearchFieldDataType, VectorSearch,
    HnswAlgorithmConfiguration
)

fields = [
    SimpleField(name="id",      type=SearchFieldDataType.String, key=True),
    SearchableField(name="title",   type=SearchFieldDataType.String),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="myHnswProfile"
    )
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="myHnsw")],
)

index = SearchIndex(name="ai-103-index", fields=fields, vector_search=vector_search)
index_client.create_or_update_index(index)
'''
print("Index-schema SDK-mönster:")
print(index_schema)

In [ ]:
# Azure AI Document Intelligence – Extrahera fält från faktura
# !pip install azure-ai-formrecognizer

def simulate_invoice_extraction() -> dict:
    """Simulerar Document Intelligence prebuilt-invoice svar."""
    return {
        "VendorName":    {"value": "Contoso AB",     "confidence": 0.99},
        "InvoiceId":     {"value": "INV-2026-0042",  "confidence": 0.98},
        "InvoiceDate":   {"value": "2026-03-15",     "confidence": 0.97},
        "DueDate":       {"value": "2026-04-14",     "confidence": 0.96},
        "InvoiceTotal":  {"value": "12 450,00 SEK",  "confidence": 0.99},
        "TaxTotal":      {"value": "2 490,00 SEK",   "confidence": 0.98},
    }

fields = simulate_invoice_extraction()

print("📄 Extraherade fakturafält:")
print("-" * 45)
for field_name, field_data in fields.items():
    bar = "█" * int(field_data["confidence"] * 10)
    print(f"  {field_name:15s}: {field_data['value']:20s} [{bar}] {field_data['confidence']:.0%}")

---
## 🔒 13. Security & MLOps
Säkerhet och driftsättning är kritiska delar av AI-103.  
Du ska kunna implementera **autentisering utan API-nycklar** och övervaka modellers prestanda.

In [ ]:
# Autentisering – Managed Identity (rekommenderat i produktion)
# !pip install azure-identity

auth_patterns = {
    "DefaultAzureCredential": {
        "use_case": "Lokalt + produktion (försöker flera metoder)",
        "code": "credential = DefaultAzureCredential()"
    },
    "ManagedIdentityCredential": {
        "use_case": "Azure-hostad applikation (VM, ACI, App Service)",
        "code": "credential = ManagedIdentityCredential()"
    },
    "ClientSecretCredential": {
        "use_case": "Service Principal (CI/CD pipelines)",
        "code": "credential = ClientSecretCredential(tenant_id, client_id, client_secret)"
    },
    "AzureKeyCredential": {
        "use_case": "API-nyckel (enklast, ej rekommenderat i produktion)",
        "code": "credential = AzureKeyCredential(api_key)"
    }
}

print("🔑 Autentiseringsalternativ:")
print("=" * 60)
for name, info in auth_patterns.items():
    print(f"\n{name}")
    print(f"  Användning: {info['use_case']}")
    print(f"  Kod: {info['code']}")

In [ ]:
# Azure Key Vault – hämta hemligheter säkert
keyvault_code = '''
from azure.keyvault.secrets import SecretClient
from azure.identity import DefaultAzureCredential

vault_url  = "https://<DITT-VAULT-NAMN>.vault.azure.net/"
credential = DefaultAzureCredential()
client     = SecretClient(vault_url=vault_url, credential=credential)

# Hämta en hemlighet (API-nyckel lagrad i Key Vault)
secret      = client.get_secret("azure-openai-key")
api_key     = secret.value  # Används sedan i SDK-anropet

# Inga API-nycklar i koden! 🔒
'''

print("🔐 Key Vault-mönster (hemligheter ur koden):")
print(keyvault_code)

In [ ]:
# Loggning och övervakning – produktionsmönster
import logging
from datetime import datetime

# Konfigurera strukturerad loggning
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("ai103.app")

def monitored_api_call(model: str, input_tokens: int, output_tokens: int):
    """Loggar AI API-anrop för övervakning."""
    start = datetime.utcnow()

    logger.info(f"API-anrop | model={model} | input_tokens={input_tokens}")

    # Simulera anrop
    latency_ms = 450
    cost_usd   = (input_tokens + output_tokens) / 1000 * 0.002

    logger.info(
        f"API-svar | model={model} | "
        f"output_tokens={output_tokens} | "
        f"latency={latency_ms}ms | "
        f"cost=${cost_usd:.4f}"
    )
    return {"tokens": output_tokens, "latency": latency_ms, "cost": cost_usd}

result = monitored_api_call("gpt-4o", input_tokens=256, output_tokens=128)
print(f"\nResultat: {result}")

In [ ]:
# Responsible AI – checklista för AI-103
responsible_ai_principles = {
    "⚖️  Fairness":        "AI-system ska behandla alla människor rättvist",
    "🔒 Reliability":     "AI-system ska prestera tillförlitligt och säkert",
    "🔐 Privacy":         "AI-system ska respektera integritet och datasäkerhet",
    "♿ Inclusiveness":   "AI-system ska ge alla människor lika möjligheter",
    "🪟 Transparency":    "AI-system ska vara begripliga och förklarbara",
    "👤 Accountability":  "Människor ska ta ansvar för AI-systemens beslut",
}

print("🤝 Microsoft Responsible AI Principles:")
print("=" * 55)
for principle, description in responsible_ai_principles.items():
    print(f"  {principle}")
    print(f"    → {description}")
    print()

---
## 📋 Sammanfattning – AI-103 Snabbreferens

| Tjänst | SDK-paket | Huvudklass |
|---|---|---|
| Azure OpenAI | `openai` | `AzureOpenAI` |
| Azure AI Vision | `azure-ai-vision-imageanalysis` | `ImageAnalysisClient` |
| Azure AI Language | `azure-ai-textanalytics` | `TextAnalyticsClient` |
| Azure AI Search | `azure-search-documents` | `SearchClient` |
| Document Intelligence | `azure-ai-formrecognizer` | `DocumentAnalysisClient` |
| Content Safety | `azure-ai-contentsafety` | `ContentSafetyClient` |
| Autentisering | `azure-identity` | `DefaultAzureCredential` |
| Key Vault | `azure-keyvault-secrets` | `SecretClient` |

### Installera alla paket på en gång:
```bash
pip install openai azure-ai-vision-imageanalysis azure-ai-textanalytics \
            azure-search-documents azure-ai-formrecognizer \
            azure-ai-contentsafety azure-identity azure-keyvault-secrets
```

---
> ©️ Stefan Blecko 2026; generated with Claude (prompt and post-production by Stefan Blecko)